# Car Sales — Preprocessing (Notebook 2)
**Team: LGTSOW**

### What is this notebook for?
Notebook 1 explored the raw data and found a bunch of problems with it (missing values,
weird data types, sketchy outliers, etc). This notebook actually **fixes** those problems
and turns the messy raw CSV into a clean table that a machine learning model can use.

We start over from the raw file — nothing is reused from Notebook 1 — so this notebook can
be run on its own from top to bottom.


### What each section below does
1. **Load the data** — read the CSV in.
2. **Dropping** — throw out columns/rows that are useless or unfixable, and explain why.
3. **Data types** — make sure every column is stored as the *right kind* of value (numbers as numbers, not text).
4. **Outliers** — deal with obviously wrong or extreme values in `year`, `price`, `mileage`.
5. **Imputation** — fill in missing values sensibly (not just "guess the average" everywhere).
6. **Feature engineering** — build new, more useful columns out of the ones we have.
7. **Encoding** — turn categories (like "Toyota", "Honda") into numbers a model can use.
8. **Keeping metadata** — keep a few original text columns around for a fairness check later.
9. **Scaling** — put all the numbers on a similar scale so no one column unfairly dominates.
10. **Split & save** — split into a training set and a testing set, and save both to disk.


## 1. Load the raw data

In [1]:
import pandas as pd
import numpy as np  
import re
import ast 
from collections import Counter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 80)   # let pandas print more columns without truncating
pd.set_option('display.width', 160)
np.random.seed(42)   
DATA_PATH = 'CarSales_Large.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
n_raw = len(df)
print(f"Raw rows: {n_raw:,}    Raw columns: {df.shape[1]}")


Raw rows: 302,590    Raw columns: 66


## 2. Dropping (getting rid of columns/rows we can't use)

### 2.1 Columns we're dropping, and why

A column is worth dropping if it can't teach the model anything useful, or if using it would
be "cheating" (leakage). Here's every column we drop and the reason:

| Column | Why it's dropped |
|---|---|
| `Unnamed: 0` | Just a leftover row-numbering column. Same info as the table's index already gives us for free. |
| `vin` | A unique ID number for each individual car. IDs don't generalize — a model can't learn "cars with VIN starting in 1FT are worth more," that's meaningless. |
| `listing_id` | Also a unique ID (for the *listing*, not the car). We use it once below to find duplicate listings, then drop it. |
| `main_picture_url` | A web link to a photo. Not something we can use as a number or category. |
| `sp_id`, `sp_name` | These identify the *dealer*. If we kept the raw dealer name/ID, the model might just memorize "this exact dealer always prices high," which won't generalize to a new dealer we've never seen. |
| `trimId` | An internal code for the trim level. We already keep `trim_name`, which says the same thing in a usable way. |
| `is_certified`, `combine_fuel_economy`, `vehicle_damage_category` | These columns are **100% empty**. There's nothing to learn from a column with no data in it at all. |
| `bed_height` | Technically has a few non-empty values, but every single one of them is just the placeholder text `"--"` — so it's empty in disguise. |
| `description` | A paragraph of dealer marketing text. Turning free text into a usable feature needs natural-language processing (NLP), which is a bigger project outside the scope of this notebook. |
| `dealer_zip` | We already have `latitude`/`longitude` for location, which are easier for a model to use directly as numbers. Keeping the ZIP code too would just be repeating the same information. |
| `transmission_display`, `wheel_system_display`, `engine_type` | These are longer, wordier versions of `transmission`, `wheel_system`, and `engine_cylinders`/`fuel_type` — same information, just messier and with way more unique values. |
| `savings_amount` | **This one is about the no-leakage rule.** This column is basically `(original price) − (price)` — it's built directly from `price`. If we kept it, the model could use simple math to "cheat" its way to a near-perfect price guess. We drop it entirely, we don't even try to build features from it. |

### 2.2 Rows we're dropping, and why

- **Exact duplicate rows** — if the same listing appears twice, keeping both copies would make the model think that listing type is twice as common as it really is.
- **Duplicate `listing_id` values** — a few listings share an ID by mistake. We keep the first one we see and drop the extras, for the same reason as above.
- **Rows with no `price`** — remember, `price` is what we're trying to predict. A row with no price is like a flashcard with no answer on the back — there's nothing to learn from it, and we can't "fill in" a missing target without just making up the answer.

In [2]:
cols_to_drop = [
    'Unnamed: 0', 'vin', 'main_picture_url', 'sp_id', 'sp_name', 'trimId',
    'is_certified', 'combine_fuel_economy', 'vehicle_damage_category', 'bed_height',
    'description', 'dealer_zip', 'transmission_display', 'wheel_system_display', 'engine_type',
    'savings_amount',
]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]   # only drop columns that actually exist
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"Shape after column drop: {df.shape}")

rows_dropped_log = {}

Dropped 15 columns: ['Unnamed: 0', 'main_picture_url', 'sp_id', 'sp_name', 'trimId', 'is_certified', 'combine_fuel_economy', 'vehicle_damage_category', 'bed_height', 'description', 'dealer_zip', 'transmission_display', 'wheel_system_display', 'engine_type', 'savings_amount']
Shape after column drop: (302590, 51)


In [3]:
# .drop_duplicates() removes rows that are 100% identical to another row
n_before = len(df)
df = df.drop_duplicates(keep='first')
rows_dropped_log['exact_duplicate_rows'] = n_before - len(df)

# Now do the same thing, but only checking the listing_id column
if 'listing_id' in df.columns:
    n_before = len(df)
    df = df.drop_duplicates(subset='listing_id', keep='first')
    rows_dropped_log['duplicate_listing_id'] = n_before - len(df)
    df = df.drop(columns=['listing_id'])   # we don't need the ID column anymore

# dropna(subset=['price']) removes any row where the price column is empty
n_before = len(df)
df = df.dropna(subset=['price'])
rows_dropped_log['missing_price_target'] = n_before - len(df)

print(rows_dropped_log)
print(f"Shape after row drops: {df.shape}")




{'exact_duplicate_rows': 3, 'duplicate_listing_id': 0, 'missing_price_target': 0}
Shape after row drops: (302587, 50)


## 3. Data types and making sure every column is stored the right way

Right now, pandas thinks some columns are just text (`object` dtype) even though they're
really numbers, dates, or yes/no flags. We need to fix that before we can do any math on them.

- **Numbers with units glued on** (like `"34.4 in"` for legroom, or `"24.6 gal"` for fuel
  tank size) — pandas sees the whole thing as one big string, not a number. We use a
  regular expression (a text-pattern search) to pull out just the number part, then
  convert it to `float` (a decimal number).
- **`power` and `torque`** are the trickiest ones — each one crams *two* pieces of
  information into a single string, like `"383 hp @ 5,600 RPM"`. We split each into two
  separate, clean numeric columns.
- **`horsepower` disagreeing with the hp inside `power`** — Notebook 1 found these two
  didn't always match. We treat the dedicated `horsepower` column as the "real" one, and
  only borrow from the `power` string when `horsepower` itself is blank.
- **True/False columns stored as text** (like `fleet`, `has_accidents`, `is_cpo`) — some of
  these have a clear rule (blank means "No"), so we fill those blanks with `False` right
  away. Others have real missing data (we genuinely don't know), so we use a special pandas
  type called **nullable boolean**, which can hold `True`, `False`, *or* "we don't know" —
  we'll properly fill those in Section 5 instead of guessing here.
- **`listed_date`** gets converted from text to an actual `datetime` type, so we can pull
  things like the month or day of week out of it later.

In [4]:
def strip_unit(series):
    # Looks for the first number (digits + optional decimal point) in each string
    # and throws away everything else (the unit label).
    return pd.to_numeric(series.astype(str).str.extract(r'([\d\.]+)')[0], errors='coerce')

unit_cols = ['back_legroom', 'front_legroom', 'fuel_tank_volume', 'height', 'length',
             'wheelbase', 'width', 'bed_length']
for col in unit_cols:
    if col in df.columns:
        df[col] = strip_unit(df[col])

if 'maximum_seating' in df.columns:
    df['maximum_seating'] = pd.to_numeric(
        df['maximum_seating'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')

print("Unit-suffixed columns are now plain numbers (float64).")


Unit-suffixed columns are now plain numbers (float64).


In [5]:
# power looks like "383 hp @ 5,600 RPM" -- pull out the hp number and the RPM number separately
if 'power' in df.columns:
    power_hp = pd.to_numeric(
        df['power'].astype(str).str.extract(r'([\d,]+)\s*hp')[0].str.replace(',', ''), errors='coerce')
    df['power_rpm'] = pd.to_numeric(
        df['power'].astype(str).str.extract(r'@\s*([\d,]+)\s*RPM')[0].str.replace(',', ''), errors='coerce')
    df = df.drop(columns=['power'])
else:
    power_hp = pd.Series(np.nan, index=df.index)

# torque works the same way, e.g. "403 lb-ft @ 3,600 RPM"
if 'torque' in df.columns:
    df['torque_lbft'] = pd.to_numeric(
        df['torque'].astype(str).str.extract(r'([\d,]+)\s*lb-ft')[0].str.replace(',', ''), errors='coerce')
    df['torque_rpm'] = pd.to_numeric(
        df['torque'].astype(str).str.extract(r'@\s*([\d,]+)\s*RPM')[0].str.replace(',', ''), errors='coerce')
    df = df.drop(columns=['torque'])

# horsepower is our "main" column -- only use the power_hp value to fill in the gaps
if 'horsepower' in df.columns:
    n_before_missing = df['horsepower'].isna().sum()
    df['horsepower'] = df['horsepower'].fillna(power_hp)
    print(f"Filled {n_before_missing - df['horsepower'].isna().sum():,} missing horsepower values from the power string")


Filled 0 missing horsepower values from the power string


In [6]:
# --- True/False columns ---
# For these columns, a blank value has a clear, known meaning ("No"), so we fill it in directly.
for col in ['is_cpo', 'is_oemcpo', 'is_new', 'franchise_dealer']:
    if col in df.columns:
        df[col] = df[col].fillna(False)

# These columns have GENUINE missing data (we truly don't know the answer) -- we use
# pandas' nullable "boolean" type so a missing value can stay marked as missing, to be
# handled properly with real imputation logic later (Section 5), instead of guessing now.
nullable_bool_cols = ['fleet', 'frame_damaged', 'has_accidents', 'isCab', 'salvage', 'theft_title']
clean_bool_cols = ['is_cpo', 'is_oemcpo', 'is_new', 'franchise_dealer']
for col in nullable_bool_cols:
    if col in df.columns:
        df[col] = df[col].astype('boolean')
for col in clean_bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

# Turn the listed_date text into an actual date pandas understands
if 'listed_date' in df.columns:
    df['listed_date'] = pd.to_datetime(df['listed_date'], errors='coerce')

print(df[[c for c in nullable_bool_cols + clean_bool_cols if c in df.columns]].dtypes)


fleet               boolean
frame_damaged       boolean
has_accidents       boolean
isCab               boolean
salvage             boolean
theft_title         boolean
is_cpo                 bool
is_oemcpo              bool
is_new                 bool
franchise_dealer       bool
dtype: object


## 4. Outliers — dealing with values that are extreme or just plain wrong

An **outlier** is a value that's way outside the normal range. Sometimes an outlier is a
real (if rare) data point — like a $500,000 supercar. Other times it's a mistake, like a
car with 99 million miles on it (that's more than 100 round trips to the moon — clearly
a typo, not a real reading). Our job is to tell those two situations apart and handle each
one appropriately, rather than treating every extreme value the same way.

- **`year`**: we drop listings from before 1980. Cars that old are usually sold as
  "collectibles," and they don't follow the normal "newer = pricier" pattern that everything
  else in the dataset does — including them would just confuse the model. We also drop any
  `year` that's suspiciously far in the future (a data-entry mistake), using the most recent
  listing date in the data as our reference point for "the present."
- **`price`**: we drop listings under \$500 — a real used car essentially never sells that
  cheap, so this is almost certainly a broken listing, not a bargain. On the *high* end, we
  don't drop anything — a $1.75 million exotic car is a real price, just a rare one. Instead
  we **winsorize** it, which just means "cap it at a high percentile instead of deleting it."
  That way the row still counts, but one extreme value can't single-handedly throw off the
  whole model.
- **`mileage`**: that 99-million-mile value is a sentinel/error value, not a real reading.
  We turn any mileage above 500,000 (already an extremely generous ceiling) into a missing
  value, and then fill it in properly using the imputation strategy in Section 5, rather
  than just deleting the whole row over one bad field.

### Checking we haven't gone overboard
The assignment requires that our combined row-dropping (missing data + outliers) never
removes more than 50% of the dataset. We've been logging every row-removal step in
`rows_dropped_log` — the code below adds it all up and uses an `assert` statement (a
built-in Python safety check that stops the notebook with an error if a condition is
false) to prove we're well under that limit.

In [7]:
if 'year' in df.columns:
    n_before = len(df)
    max_listed_year = df['listed_date'].dt.year.max() if 'listed_date' in df.columns else df['year'].max()
    df = df[(df['year'] >= 1980) & (df['year'] <= max_listed_year + 1)]
    rows_dropped_log['year_out_of_range'] = n_before - len(df)

if 'price' in df.columns:
    n_before = len(df)
    df = df[df['price'] >= 500]
    rows_dropped_log['price_under_500'] = n_before - len(df)

    # winsorizing: cap (don't delete) values above the 99.5th percentile
    price_cap = df['price'].quantile(0.995)
    n_capped = (df['price'] > price_cap).sum()
    df['price'] = df['price'].clip(upper=price_cap)
    print(f"Capped {n_capped:,} price values above the 99.5th percentile (${price_cap:,.0f})")

if 'mileage' in df.columns:
    n_sentinel = (df['mileage'] > 500_000).sum()
    df.loc[df['mileage'] > 500_000, 'mileage'] = np.nan   # mark it missing instead of deleting the row
    print(f"Recoded {n_sentinel} impossible mileage value(s) to missing (to be filled in later)")

print(rows_dropped_log)


Capped 1,512 price values above the 99.5th percentile ($110,892)
Recoded 2 impossible mileage value(s) to missing (to be filled in later)
{'exact_duplicate_rows': 3, 'duplicate_listing_id': 0, 'missing_price_target': 0, 'year_out_of_range': 277, 'price_under_500': 1}


In [8]:
total_dropped = sum(rows_dropped_log.values())
pct_dropped = total_dropped / n_raw * 100
print(f"Total rows dropped so far: {total_dropped:,} out of {n_raw:,} raw rows = {pct_dropped:.2f}%")

# This line will literally stop the notebook with an error if we ever crossed 50% --
# a safety net so we can't accidentally throw away too much data.
assert pct_dropped <= 50, "We dropped more than 50% of the rows -- something's wrong!"
print("Confirmed: we're well under the 50% limit.")
print(f"Rows remaining: {len(df):,}")


Total rows dropped so far: 281 out of 302,590 raw rows = 0.09%
Confirmed: we're well under the 50% limit.
Rows remaining: 302,309


## 5. Imputation (filling in the blanks)

**Imputation** just means "filling in a missing value with a reasonable guess" instead of
leaving it blank or throwing the whole row away. 

So for each column below, we picked a fill in strategy based on what would actually make
sense for that specific kind of data:

| Column(s) | How we fill it in | Why |
|---|---|---|
| `mileage` | `0` if the listing is brand new, otherwise the typical mileage for cars of that same `year` | New cars really do have ~0 miles, and mileage mostly comes down to how old the car is. |
| `owner_count` | Same idea: `0` if brand new, otherwise the typical owner count for that `year` | A new car has had zero previous owners; older cars tend to have had more. |
| `horsepower`, `engine_displacement`, `torque_lbft`, `torque_rpm`, `power_rpm` | The typical value for that exact `make` + `model` (falls back to the overall typical value only if that model has no data at all) | Every Honda Civic has basically the same engine. |
| Size columns (`legroom`, `height`, `length`, `width`, `wheelbase`, `fuel_tank_volume`, `maximum_seating`) | The typical value for that `body_type` (like "SUV," "Sedan") | Cars of the same body type are a similar size, regardless of brand a sedan and a pickup truck are very different sizes no matter who makes them. |
| `bed_length` | `0` for anything that isn't a pickup truck (it doesn't have a bed), the typical pickup bed length otherwise | This isn't really "missing" data, a sedan doesn't have a truck bed, so `0`/not-applicable is the *correct* answer. |
| `bed`, `cabin` | `"Not Applicable"` | these only make sense for trucks in the first place. |
| `city_fuel_economy`, `highway_fuel_economy` | The typical value for that `fuel_type` | Electric, gas, and hybrid cars have wildly different fuel-economy numbers , grouping by fuel type avoids putting a gas car mpg onto a missing EV row. |
| `franchise_make` | `"Independent"` | A blank here almost always just means the dealer isn't tied to a specific manufacturer brand , that's useful information on its own, not a guess. |
| `interior_color`, `exterior_color` | `"Unknown"` | There's no good way to guess a car's color from other columns, so we're honest about not knowing rather than picking the most common color and pretending we know. |
| `seller_rating` | The typical rating for that `city` | Dealer ratings plausibly cluster or change by local market. |
| `listed_date` | The middle date in the dataset | Very few of these are missing, so a safe default has little impact either way. |
| `major_options` | Treated as "no extra options were listed" (an empty list), not truly missing | A blank options field means the dealer didn't add any listed extras ,that's a real answer, not missing data. |
| `fleet`, `frame_damaged`, `has_accidents`, `salvage`, `theft_title`, `isCab` | We first add a "was this missing?" flag column, *then* fill the blank with `False` | About 46% of listings are missing a vehicle history report. No one knows if the cars are clean or not. Filling with `False` (no known issue) is a reasonable default since `False` is by far the most common *known* answer, but the flag column lets the model still tell "verified clean" apart from "we just don't know," instead of us silently pretending we know the answer. |


In [9]:
# --- mileage & owner_count ---
is_new_bool = df['is_new'].astype('boolean').fillna(False) if 'is_new' in df.columns else pd.Series(False, index=df.index)

if 'mileage' in df.columns:
    df['mileage_missing'] = df['mileage'].isna().astype(int)   # flag: 1 if we had to fill this in, else 0
    year_median_mileage = df.groupby('year')['mileage'].transform('median')  # typical mileage per model year
    df.loc[df['mileage'].isna() & is_new_bool, 'mileage'] = 0   # new cars: assume 0 miles
    df['mileage'] = df['mileage'].fillna(year_median_mileage)   # everyone else: typical mileage for that year
    df['mileage'] = df['mileage'].fillna(df['mileage'].median())  # last resort safety net

if 'owner_count' in df.columns:
    df['owner_count_missing'] = df['owner_count'].isna().astype(int)
    year_median_owners = df.groupby('year')['owner_count'].transform('median')
    df.loc[df['owner_count'].isna() & is_new_bool, 'owner_count'] = 0
    df['owner_count'] = df['owner_count'].fillna(year_median_owners)
    df['owner_count'] = df['owner_count'].fillna(df['owner_count'].median())

print("mileage / owner_count filled in.")


mileage / owner_count filled in.


In [10]:
# --- mechanical specs: use the typical value for that exact make + model ---
mech_cols = ['horsepower', 'engine_displacement', 'torque_lbft', 'torque_rpm', 'power_rpm']
if all(c in df.columns for c in ['make_name', 'model_name']):
    for col in mech_cols:
        if col in df.columns:
            df[f'{col}_missing'] = df[col].isna().astype(int)
            # .groupby(...).transform('median') computes the median WITHIN each make+model
            # group, and lines it back up with the original rows -- so every row gets
            # "the typical value for cars just like this one."
            group_median = df.groupby(['make_name', 'model_name'])[col].transform('median')
            df[col] = df[col].fillna(group_median)
            df[col] = df[col].fillna(df[col].median())   # fallback if a whole model has no data

print("Mechanical spec columns filled in using make/model typical values.")


Mechanical spec columns filled in using make/model typical values.


In [11]:
# --- physical size columns: use the typical value for that body type ---
dim_cols = ['back_legroom', 'front_legroom', 'fuel_tank_volume', 'height', 'length',
            'wheelbase', 'width', 'maximum_seating']
if 'body_type' in df.columns:
    for col in dim_cols:
        if col in df.columns:
            group_median = df.groupby('body_type')[col].transform('median')
            df[col] = df[col].fillna(group_median)
            df[col] = df[col].fillna(df[col].median())

print("Size columns filled in using body-type typical values.")


Size columns filled in using body-type typical values.


In [12]:
# --- bed_length / bed / cabin: missing usually means "doesn't apply" (not a truck) ---
if 'body_type' in df.columns:
    is_pickup = df['body_type'] == 'Pickup Truck'
    if 'bed_length' in df.columns:
        df.loc[~is_pickup & df['bed_length'].isna(), 'bed_length'] = 0        # not a truck -> 0, correctly
        pickup_median = df.loc[is_pickup, 'bed_length'].median()
        df.loc[is_pickup & df['bed_length'].isna(), 'bed_length'] = pickup_median
    for col in ['bed', 'cabin']:
        if col in df.columns:
            df[col] = df[col].fillna('Not Applicable')

print("bed_length / bed / cabin filled in.")


bed_length / bed / cabin filled in.


In [13]:
# --- fuel economy: use the typical value for that fuel type ---
if 'fuel_type' in df.columns:
    for col in ['city_fuel_economy', 'highway_fuel_economy']:
        if col in df.columns:
            group_median = df.groupby('fuel_type')[col].transform('median')
            df[col] = df[col].fillna(group_median)
            df[col] = df[col].fillna(df[col].median())

# --- categorical columns where "missing" has an actual meaning ---
if 'franchise_make' in df.columns:
    df['franchise_make'] = df['franchise_make'].fillna('Independent')
for col in ['interior_color', 'exterior_color']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# --- seller_rating: use the typical rating for that city ---
if 'seller_rating' in df.columns and 'city' in df.columns:
    group_median = df.groupby('city')['seller_rating'].transform('median')
    df['seller_rating'] = df['seller_rating'].fillna(group_median)
    df['seller_rating'] = df['seller_rating'].fillna(df['seller_rating'].median())

# --- listed_date: use the middle date ---
if 'listed_date' in df.columns:
    df['listed_date'] = df['listed_date'].fillna(df['listed_date'].median())

print("Fuel economy / colors / franchise_make / seller_rating / listed_date filled in.")


Fuel economy / colors / franchise_make / seller_rating / listed_date filled in.


In [14]:
# --- vehicle-history columns: flag that it was missing, THEN fill with False ---
history_cols = ['fleet', 'frame_damaged', 'has_accidents', 'salvage', 'theft_title', 'isCab']
for col in history_cols:
    if col in df.columns:
        df[f'{col}_missing'] = df[col].isna().astype(int)   # 1 = "we don't actually know", 0 = "we know"
        df[col] = df[col].fillna(False).astype(bool)

print("Vehicle-history columns: missing values flagged, then filled with False.")
print()

# Sanity check -- is anything still missing after all of this?
remaining_na = df.isna().sum()
print("Columns that STILL have missing values (should be empty/none):")
print(remaining_na[remaining_na > 0].to_string() if remaining_na.sum() > 0 else "None — every missing value has been filled in!")


Vehicle-history columns: missing values flagged, then filled with False.

Columns that STILL have missing values (should be empty/none):
body_type            1339
engine_cylinders    10213
fuel_type            8309
major_options       18335
transmission         7545
trim_name           11164
wheel_system        14373


## 6. Feature engineering (building new, more useful columns)

**Feature engineering** means creating brand-new columns out of the ones we already have,
to make patterns easier for a model to pick up on. For example, "car's age" is much easier
for a model to use directly than making it figure out "newer year = younger car" on its own
from the raw `year` number.

Reminder: everything below is built only from `year`, `listed_date`, `mileage`,
`engine_cylinders`, `major_options`, `make_name`, and the vehicle history flags — **never**
from `price`.

- **`vehicle_age`** = (the year the car was listed) − (the car's model `year`). More directly
  useful than the raw year, and it's calculated from data we actually have (not "today's
  date," which wouldn't be reproducible).
- **`mileage_per_year`** = `mileage` ÷ `vehicle_age`. 
- **`listing_month`, `listing_dayofweek`** — pulled straight out of `listed_date`
- **`cylinder_count`, `engine_layout`** — `engine_cylinders` comes in as a string like
  `"V6"` (6 cylinders, "V" layout). We split that into a number (`6`) and a category
  (`"V"`), so both parts are usable separately. Electric cars have no cylinders at all, so
  they get `cylinder_count = 0` and `engine_layout = "Electric"`.
- **`n_major_options` and the `opt_*` flag columns** — `major_options` arrives as a string
  that literally looks like Python code, e.g. `"['Leather Seats', 'Navigation System']"`.
  We use `ast.literal_eval()` (a safe way to turn that text back into a real Python list)
  to parse it, count how many options each listing has, and create a `True`/`False` column
  for each of the 20 most common individual options (a blank field just means "no options
  were listed," not missing data).
- **`has_history_issue`** — one combined flag that's `True` if *any* of frame damage,
  accident history, salvage title, or theft title is `True`. Bundling these together gives
  the model one simple "red flag" signal instead of four separate ones.
- **`is_luxury_make`** — `True` if the brand is a well-known luxury brand (BMW,
  Mercedes-Benz, Audi, Lexus, Porsche, etc). Brand prestige is a real driver of price that
  isn't fully captured by horsepower/size alone

In [15]:
if 'listed_date' in df.columns and 'year' in df.columns:
    # .clip(lower=0) just means "don't let this go below zero"
    df['vehicle_age'] = (df['listed_date'].dt.year - df['year']).clip(lower=0)
    df['listing_month'] = df['listed_date'].dt.month
    df['listing_dayofweek'] = df['listed_date'].dt.dayofweek

if 'mileage' in df.columns and 'vehicle_age' in df.columns:
    # clip(lower=1) avoids dividing by zero for brand-new cars
    df['mileage_per_year'] = df['mileage'] / df['vehicle_age'].clip(lower=1)

print("Date/age-based features created.")


Date/age-based features created.


In [16]:
def parse_cylinders(val):
    """Turn a string like 'V6' into (6, 'V'). Electric cars have no cylinders."""
    if pd.isna(val):
        return (np.nan, 'Unknown')
    s = str(val)
    if 'lectric' in s or 'Electric' in s:
        return (0, 'Electric')
    m = re.match(r'([A-Za-z]+)\s*(\d+)', s)   # letters, then digits (e.g. "V" then "6")
    if m:
        layout, count = m.group(1), int(m.group(2))
        return (count, layout)
    return (np.nan, 'Unknown')

if 'engine_cylinders' in df.columns:
    parsed = df['engine_cylinders'].apply(parse_cylinders)
    df['cylinder_count'] = parsed.apply(lambda t: t[0])
    df['cylinder_count'] = df['cylinder_count'].fillna(df['cylinder_count'].median())
    df['engine_layout'] = parsed.apply(lambda t: t[1])
    df = df.drop(columns=['engine_cylinders'])   # replaced by the two cleaner columns above

print("cylinder_count / engine_layout created from engine_cylinders.")


cylinder_count / engine_layout created from engine_cylinders.


In [17]:
def parse_options(val):
    """Safely turn a string like "['Leather Seats', 'Navigation']" into a real Python list."""
    if pd.isna(val):
        return []   # no options listed
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []   # if the text is broken/unreadable, just treat it as "no options"

if 'major_options' in df.columns:
    options_parsed = df['major_options'].apply(parse_options)
    df['n_major_options'] = options_parsed.apply(len)   # how many options each listing has

    # Count how often each individual option shows up across the whole dataset,
    # then keep only the 20 most common ones as their own True/False columns.
    option_counts = Counter(opt for opts in options_parsed for opt in opts)
    TOP_N_OPTIONS = 20
    top_options = [opt for opt, _ in option_counts.most_common(TOP_N_OPTIONS)]

    for opt in top_options:
        col_name = 'opt_' + re.sub(r'[^0-9a-zA-Z]+', '_', opt).strip('_').lower()
        df[col_name] = options_parsed.apply(lambda opts, o=opt: o in opts)

    df = df.drop(columns=['major_options'])
    print(f"Parsed major_options into n_major_options + {len(top_options)} True/False option columns.")


Parsed major_options into n_major_options + 20 True/False option columns.


In [18]:
if all(c in df.columns for c in ['frame_damaged', 'has_accidents', 'salvage', 'theft_title']):
    # the | symbol means "OR" -- this is True if ANY of the four flags is True
    df['has_history_issue'] = (
        df['frame_damaged'].astype(bool) | df['has_accidents'].astype(bool)
        | df['salvage'].astype(bool) | df['theft_title'].astype(bool)
    )

LUXURY_MAKES = {
    'BMW', 'Mercedes-Benz', 'Audi', 'Lexus', 'Porsche', 'Jaguar', 'Land Rover', 'Maserati',
    'Bentley', 'Rolls-Royce', 'Ferrari', 'Lamborghini', 'McLaren', 'Bugatti', 'Tesla',
    'Cadillac', 'Genesis', 'INFINITI', 'Infiniti', 'Acura', 'Alfa Romeo', 'Aston Martin',
}
if 'make_name' in df.columns:
    # .isin(...) checks each row's make_name against our list of luxury brands
    df['is_luxury_make'] = df['make_name'].isin(LUXURY_MAKES)

print("has_history_issue / is_luxury_make created.")
print(f"Shape after feature engineering: {df.shape}")


has_history_issue / is_luxury_make created.
Shape after feature engineering: (302309, 91)


## 7. Encoding (turning categories into numbers)

### Binary (True/False) columns need → label encoding
This is the easy case: a column that's just `True`/`False` can be converted straight to
`1`/`0`. 
### 7.2 Category columns with more than 2 options → one-hot encoding
For a column like `make_name` (Toyota, Honda, Ford, etc), we can't just number the brands
1, 2, 3 because that would falsely imply Ford (say, `3`) is somehow "more" than Toyota
(`1`), there's no real order between car brands. Instead, **one-hot encoding** creates a
separate True/False (1/0) column for each category,like `make_name_Toyota`,
`make_name_Honda`, one of which is `1` and the rest are `0` for any given row.


## 8. Keeping some original columns around (metadata)
For every column we one-hot encode, we keep the **original, unencoded version** sitting
right next to its new numeric columns ( `make_name` stays right next to
`make_name_Toyota`, `make_name_Honda`, etc).

Later, right before we save the final train/test CSVs (Section 10), we'll rename each of
these retained originals with a **`meta_` prefix** (`make_name` -> `meta_make_name`). This
way, later notebooks can find and exclude every metadata column just by checking the
column name -- `c.startswith('meta_')` -- instead of needing to know (or re-derive) the
exact list of which columns are metadata.

In [19]:
# pd.api.types.is_bool_dtype() checks "is this column made of True/False values?"
# str(dtype) == 'boolean' catches pandas' special *nullable* boolean type too.
# Scanning for ALL boolean columns (instead of typing out a fixed list) means every
# engineered flag from Section 6 -- the opt_* columns, is_luxury_make, etc. -- gets
# picked up automatically, so nothing slips through un-encoded.
binary_cols = [c for c in df.columns
               if pd.api.types.is_bool_dtype(df[c]) or str(df[c].dtype) == 'boolean']

for col in binary_cols:
    df[col] = df[col].astype(bool).astype(int)   # True/False -> 1/0

print(f"Label-encoded {len(binary_cols)} binary columns:")
print(binary_cols)


Label-encoded 32 binary columns:
['fleet', 'frame_damaged', 'franchise_dealer', 'has_accidents', 'isCab', 'is_cpo', 'is_new', 'is_oemcpo', 'salvage', 'theft_title', 'opt_backup_camera', 'opt_bluetooth', 'opt_alloy_wheels', 'opt_heated_seats', 'opt_navigation_system', 'opt_sunroof_moonroof', 'opt_carplay', 'opt_remote_start', 'opt_leather_seats', 'opt_android_auto', 'opt_blind_spot_monitoring', 'opt_parking_sensors', 'opt_adaptive_cruise_control', 'opt_third_row_seating', 'opt_steel_wheels', 'opt_quick_order_package', 'opt_premium_package', 'opt_convenience_package', 'opt_tow_package', 'opt_multi_zone_climate_control', 'has_history_issue', 'is_luxury_make']


In [20]:
def one_hot_top_n(df, col, n=20):
    """
    One-hot encode a category column, but keep it under control:
      - the MOST common category becomes the "default" (no column of its own)
      - the next `n` most common categories each get their own 1/0 column
      - anything rarer just falls into the default bucket (all zeros)
    """
    value_counts = df[col].value_counts()      # counts how many rows have each category, biggest first
    reference_cat = value_counts.index[0]      # the single most common category
    top_n_cats = value_counts.index[1:n + 1]   # the next n most common categories

    dummies = pd.DataFrame(index=df.index)
    for cat in top_n_cats:
        dummy_name = f"{col}_{re.sub(r'[^0-9a-zA-Z]+', '_', str(cat)).strip('_')}"
        dummies[dummy_name] = (df[col] == cat).astype(int)   # 1 if this row matches, else 0
    return dummies, reference_cat

categorical_cols_to_encode = [c for c in [
    'make_name', 'model_name', 'body_type', 'fuel_type', 'transmission', 'wheel_system',
    'listing_color', 'city', 'franchise_make', 'exterior_color', 'interior_color',
    'trim_name', 'bed', 'cabin', 'engine_layout',
] if c in df.columns]

metadata_cols = []
all_dummies = []
for col in categorical_cols_to_encode:
    dummies, reference_cat = one_hot_top_n(df, col, n=20)
    all_dummies.append(dummies)
    metadata_cols.append(col)   # remember this column's original version needs to be kept
    print(f"{col}: default category = '{reference_cat}', {dummies.shape[1]} new columns created")

# pd.concat glues all the new dummy-column tables onto the original dataframe, side by side
df = pd.concat([df] + all_dummies, axis=1)
print(f"\nShape after one-hot encoding: {df.shape}")
print(f"Original columns kept for later (fairness audit), not used for modeling: {metadata_cols}")


make_name: default category = 'Ford', 20 new columns created
model_name: default category = 'F-150', 20 new columns created
body_type: default category = 'SUV / Crossover', 8 new columns created
fuel_type: default category = 'Gasoline', 6 new columns created
transmission: default category = 'A', 3 new columns created
wheel_system: default category = 'FWD', 4 new columns created
listing_color: default category = 'WHITE', 14 new columns created
city: default category = 'Houston', 20 new columns created
franchise_make: default category = 'Independent', 20 new columns created
exterior_color: default category = 'Black', 20 new columns created
interior_color: default category = 'Black', 20 new columns created
trim_name: default category = 'SE FWD', 20 new columns created
bed: default category = 'Not Applicable', 3 new columns created
cabin: default category = 'Not Applicable', 4 new columns created
engine_layout: default category = 'I', 5 new columns created

Shape after one-hot encoding: (3

## 9. Scaling — putting every number on the same footing

Right now our numeric columns are on wildly different scales — `mileage` might be in the
tens of thousands, while `n_major_options` only ranges from 0 to about 20. Some models get
confused by this and end up treating the *bigger-looking* numbers as automatically more
important, even when they're not.

**Standard-scaling** fixes this by converting every column to "how many standard deviations
away from the average is this value" — after scaling, every column has a mean of 0 and a
similar spread, so they're all being compared fairly.

We scale every feature **except**:
- the `metadata_cols` from Section 8 (they're just kept as reference text, not model input)
- `price`, which needs to stay in real dollars since it's our prediction target

In [21]:
exclude_from_scaling = set(metadata_cols) | {'price'}
if 'listed_date' in df.columns:
    exclude_from_scaling.add('listed_date')   # it's a date, not a plain number, so scaling doesn't apply

# only scale columns that are (a) not in our exclude list, and (b) actually numeric
scale_cols = [c for c in df.columns
              if c not in exclude_from_scaling and pd.api.types.is_numeric_dtype(df[c])]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print(f"Standard-scaled {len(scale_cols)} feature columns.")
print(f"price was left alone — still in real dollars, from {df['price'].min():,.0f} to {df['price'].max():,.0f}")


Standard-scaled 261 feature columns.
price was left alone — still in real dollars, from 670 to 110,892


## 10. Split & export — saving our work

Before we can train and test a model, we need to split the data into two separate piles:

- **Training set (80%)** — the data the model actually learns from.
- **Testing set (20%)** — data the model never sees while learning, held back so we can
  check afterward how well it does on "new" listings it hasn't memorized.

`train_test_split(..., random_state=42)` does this split for us. Setting `random_state=42`
just means "use the same random shuffle every time we run this" — that way our split is
reproducible, and Notebook 3 can rebuild the *exact same* train/test split rather than
getting a different random mix each time.

Both files keep every column, including `price` (the answer we're trying to predict) and
`metadata_cols` (needed for the Notebook 5 fairness check) — it'll be up to whichever
notebook actually trains a model to separate those out first.

Right before saving, we rename every `metadata_cols` column with a **`meta_` prefix**
(`make_name` -> `meta_make_name`). We wait until this last step rather than renaming
earlier, so every cell above this one can keep referring to columns by their original,
familiar names — only the two exported CSVs (what later notebooks actually read) need
the prefix.

In [22]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"Training set: {train_df.shape[0]:,} rows")
print(f"Testing set:  {test_df.shape[0]:,} rows")

# Prefix the retained metadata columns with "meta_" -- but only now, right before saving.
# Future notebooks can then find and exclude them by name pattern alone (e.g. `[c for c
# in df.columns if not c.startswith('meta_')]`), without needing to know this specific
# column list. Only the original categorical columns get renamed -- the one-hot dummy
# columns (e.g. `make_name_Toyota`) keep their plain names, since those ARE model features.
rename_map = {col: f'meta_{col}' for col in metadata_cols}
train_df = train_df.rename(columns=rename_map)
test_df = test_df.rename(columns=rename_map)
metadata_cols = [rename_map[col] for col in metadata_cols]

OUT_DIR = '.'   # <-- change this to point at your local class data folder
train_path = f'{OUT_DIR}/processed_train.csv'
test_path = f'{OUT_DIR}/processed_test.csv'

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved: {train_path}")
print(f"Saved: {test_path}")
print()
print("Reminder for Notebook 3: build X by dropping ['price'] + metadata_cols; y = df['price'].")
print(f"metadata_cols = {metadata_cols}")


Training set: 241,847 rows
Testing set:  60,462 rows
Saved: ./processed_train.csv
Saved: ./processed_test.csv

Reminder for Notebook 3: build X by dropping ['price'] + metadata_cols; y = df['price'].
metadata_cols = ['meta_make_name', 'meta_model_name', 'meta_body_type', 'meta_fuel_type', 'meta_transmission', 'meta_wheel_system', 'meta_listing_color', 'meta_city', 'meta_franchise_make', 'meta_exterior_color', 'meta_interior_color', 'meta_trim_name', 'meta_bed', 'meta_cabin', 'meta_engine_layout']
